# Setup and data

This notebook downloads the canonical six-ticker basket used throughout the
rest of the notebook series and caches it to disk so the other notebooks can
load the data instantly without hitting the network.

The basket is:

- `SPY` — US large-cap equity
- `TLT` — long-duration US Treasury
- `AAPL` — single-name equity
- `KO` — defensive single-name equity
- `BTC-USD` — crypto
- `^VIX` — volatility index, used as a feature in later notebooks

The sample period is 2015-01-01 to 2024-12-31, which covers two full bear-
to-bull cycles in equities and crypto. Run this notebook once; the other
notebooks read from the parquet cache.

Outputs written by this notebook:

- `data/cache/basket_returns.parquet`
- `data/cache/basket_rf.parquet`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from riskmetrics.data import compute_returns, get_prices, get_risk_free

In [ ]:
TICKERS = ["SPY", "TLT", "AAPL", "KO", "BTC-USD", "^VIX"]
START = "2015-01-01"
END = "2024-12-31"

prices = get_prices(TICKERS, start=START, end=END)
prices.tail()

In [ ]:
returns = compute_returns(prices, method="simple")
print("shape:", returns.shape)
returns.head()

In [ ]:
returns.describe()

In [ ]:
rf = get_risk_free(start=START, end=END)
print("shape:", rf.shape)
rf.head()

In [ ]:
# Annualised risk-free rate over the sample (mean of daily series * 252)
annualised_rf = float(rf.mean()) * 252
print(f"Mean annualised risk-free rate over {START} to {END}: {annualised_rf:.4%}")

In [ ]:
# Persist the basket so the other notebooks can load it without re-downloading.
cache_dir = Path("data/cache")
cache_dir.mkdir(parents=True, exist_ok=True)

returns_path = cache_dir / "basket_returns.parquet"
rf_path = cache_dir / "basket_rf.parquet"

returns.to_parquet(returns_path)
rf.to_frame(name="rf").to_parquet(rf_path) if isinstance(rf, pd.Series) else rf.to_parquet(rf_path)

print(f"Wrote {returns_path} ({returns_path.stat().st_size / 1024:.1f} KB)")
print(f"Wrote {rf_path} ({rf_path.stat().st_size / 1024:.1f} KB)")

Next: see `01_returns_and_distributions.ipynb` (to be added).